# Model Optimization

This notebook applies post-training optimization techniques to reduce model size and improve inference speed for embedded deployment.

## What This Notebook Does

1. **Quantization**: Converts model weights from float32 to int8/int4, reducing size by 75-90%
2. **ONNX Export**: Converts PyTorch model to ONNX format for cross-platform deployment and optimization
3. **Benchmarking**: Measures inference latency and throughput to validate performance targets
4. **Artifact Management**: Saves optimized models and generates summary reports with size/performance metrics

## Why Optimization Matters

**Size Reduction:**
- Original model: ~2-5GB (float32 weights)
- Quantized (int8): ~500MB-1.25GB (75% reduction)
- Quantized (int4): ~250MB-625MB (87.5% reduction)

**Speed Improvements:**
- Reduced memory bandwidth requirements
- CPU-optimized inference operators
- Better cache utilization on embedded hardware

**Target Hardware:**
- Embedded Linux devices with <4GB RAM
- CPU-only inference (no GPU available)
- <50ms latency requirement for interactive use

## Optimization Techniques

### Quantization
Reduces precision of model weights and activations while maintaining accuracy:
- **int8**: Good balance of size/accuracy, ~1% accuracy loss
- **int4**: Maximum size reduction, ~3-5% accuracy loss

### ONNX Runtime
Cross-platform inference engine with optimizations:
- Operator fusion (combine multiple ops into one)
- Constant folding (pre-compute static values)
- Memory layout optimization
- CPU-specific optimizations (AVX2, AVX512)

## Expected Results

| Metric | Original | Quantized (int8) | ONNX Runtime |
|--------|----------|------------------|--------------|
| Size | 2-5GB | 500MB-1.25GB | Similar to PyTorch |
| Latency (CPU) | 200-500ms | 50-150ms | 40-120ms |
| Memory (peak) | 8-12GB | 2-4GB | 1.5-3GB |

## Prerequisites

- Completed notebook `07-download-trained-model.ipynb`
- Trained model available in `models/` directory
- Sufficient disk space (~10GB) for optimization artifacts

## Expected Duration

~15-30 minutes depending on model size and hardware

In [ ]:
# 1. Import Libraries
import json
import time
from pathlib import Path
from dataclasses import dataclass
import logging
from typing import List, Dict, Any

from src.training.model_optimizer import (
    quantize_model,
    export_to_onnx,
    summarize_artifact,
    benchmark_inference,
)


**Why these imports?** The `model_optimizer` module contains pre-built functions for quantization, ONNX export, benchmarking, and artifact summarization. We reuse these tested functions rather than writing optimization code from scratch.

In [ ]:
# 2. Load Optimization Config
config_path = Path("configs/optimization_config.yaml")
if not config_path.exists():
    raise FileNotFoundError(f"Missing config: {config_path}")

import yaml
with open(config_path) as f:
    raw_cfg = yaml.safe_load(f)
raw_cfg

**Why load config?** The optimization config specifies quantization method (int8/int4), ONNX opset version, benchmark parameters, and output paths. This keeps optimization settings separate from code.

In [ ]:
# 3. Quantize Model
base_model_id = raw_cfg["base_model_id"]
quant_cfg = raw_cfg["quantization"]
quant_dir = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}-quant"

if quant_cfg.get("enabled", True):
    quant_result = quantize_model(
        base_model_id,
        quant_dir,
        method=quant_cfg.get("method", "int8"),
        dtype=quant_cfg.get("dtype"),
    )
    quant_meta = summarize_artifact(quant_result)
else:
    quant_result = None
    quant_meta = {"enabled": False}
quant_meta

**Why quantize?** Quantization converts float32 weights to lower precision (int8/int4), dramatically reducing model size and improving CPU inference speed with minimal accuracy loss. This is critical for embedded deployment.

In [ ]:
# 4. Export ONNX
onnx_cfg = raw_cfg["onnx"]
onnx_path = Path(raw_cfg["output_root"]) / f"{base_model_id.replace('/', '_')}.onnx"
if onnx_cfg.get("enabled", True):
    onnx_result = export_to_onnx(
        base_model_id,
        onnx_path,
        opset=onnx_cfg.get("opset", 17),
        sequence_length=onnx_cfg.get("sequence_length", 128),
    )
    onnx_meta = summarize_artifact(onnx_result)
else:
    onnx_meta = {"enabled": False}
onnx_meta

**Why ONNX?** ONNX Runtime provides cross-platform inference with hardware-specific optimizations (CPU vectorization, memory layout). It's faster than raw PyTorch on CPU and enables deployment on non-Python environments.

In [ ]:
# 5. Benchmark Quantized Artifact (if present)
if quant_result:
    bench_stats = benchmark_inference(quant_result.artifact_path, repetitions=raw_cfg["benchmark"]["repetitions"], max_new_tokens=raw_cfg["benchmark"]["max_new_tokens"], prompt=raw_cfg["benchmark"]["prompt"])
else:
    bench_stats = {"skipped": True}
bench_stats

**Why benchmark?** We measure actual inference latency on CPU hardware to validate that optimizations meet the <50ms target. Benchmarking reveals if further optimization or hardware upgrades are needed.

In [ ]:
# 6. Compare Sizes
sizes = {}
if quant_result:
    sizes["quantized_bytes"] = quant_result.size_bytes
if onnx_path.exists():
    sizes["onnx_bytes"] = onnx_path.stat().st_size
sizes

In [ ]:
# 7. Persist Summary
summary = {
    "quantization": quant_meta,
    "onnx": onnx_meta,
    "benchmark": bench_stats,
    "sizes": sizes,
}
summary_path = Path(raw_cfg["output_root"]) / "summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2))
summary_path, summary

**Why save summary?** The summary JSON captures all optimization metrics (sizes, latencies, configurations) for later analysis, documentation, and deployment decisions. It serves as an optimization audit trail.